# VeriForgot — Notebook 03: MIA Oracle Evaluation

Runs the full MIA oracle on all trained models.
Produces Table 2 and Figure 2 from the CRBL 2026 paper.

**Requires:** `model_original.pth`, `model_ga.pth`, `model_kd.pth` from Notebooks 01 & 02.

In [ ]:
import torch, torch.nn as nn
import torchvision, torchvision.transforms as transforms
import torchvision.models as models
import numpy as np, pickle, json
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Subset

SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion_per = nn.CrossEntropyLoss(reduction='none')
print(f'Device: {device}')

In [ ]:
# ── Load data ─────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
full_eval = torchvision.datasets.CIFAR10('./data', True,  download=True, transform=transform_test)
testset   = torchvision.datasets.CIFAR10('./data', False, download=True, transform=transform_test)

with open('forget_indices.pkl','rb') as f: forget_indices = pickle.load(f)

# Non-members: test set classes 2-9 only
nm_indices = [i for i,(_, l) in enumerate(testset) if l not in [0,1]][:500]

forget_eval_ld    = DataLoader(Subset(full_eval, forget_indices), batch_size=256, shuffle=False, num_workers=2)
non_member_loader = DataLoader(Subset(testset,   nm_indices),     batch_size=256, shuffle=False, num_workers=2)
print(f'Forget eval: {len(forget_indices)} | Non-members: {len(nm_indices)}')

In [ ]:
# ── MIA Oracle ────────────────────────────────────────
TAU = 0.57   # Oracle threshold

def load_model(path):
    m = models.resnet18(weights=None); m.fc = nn.Linear(512,10)
    m.load_state_dict(torch.load(path, map_location=device))
    return m.to(device).eval()

def get_conf(model, loader):
    s = []
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            p=torch.softmax(model(x),dim=1)
            s.extend(p[torch.arange(len(y)),y].cpu().numpy())
    return np.array(s)

def run_mia(model, name=''):
    mc = get_conf(model, forget_eval_ld)
    nc = get_conf(model, non_member_loader)
    labels = np.concatenate([np.ones(len(mc)), np.zeros(len(nc))])
    auc = roc_auc_score(labels, np.concatenate([mc, nc]))
    verdict = 'PASS' if auc < TAU else 'FAIL'
    print(f'[{name}]  AUC={auc:.4f}  ConfGap={np.mean(mc)-np.mean(nc):.4f}  {verdict}')
    return auc, verdict, mc, nc

model_paths = {
    'Original'         : '/kaggle/working/model_original.pth',
    'Gradient Ascent'  : '/kaggle/working/model_ga.pth',
    'Selective Retrain': '/kaggle/working/model_kd.pth',
}

results = {}
for name, path in model_paths.items():
    auc, verdict, mc, nc = run_mia(load_model(path), name)
    results[name] = {'auc': auc, 'verdict': verdict}